In [7]:
!pip install  panda_gym

In [3]:
import gymnasium as gym 
import panda_gym 
from stable_baselines3 import TD3
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv

In [4]:
# ==============================================================================
# 1. Environment Setup
# ==============================================================================
# Define the environment ID.
env_id = 'PandaPickAndPlace-v3'

# Create a vectorized environment. This is crucial for efficient training
# as it allows the agent to interact with multiple environments simultaneously.
# The `wrapper_class` is set to None as we don't need a specific wrapper here.
# TD3 is often more stable and robust for continuous control tasks than DDPG.
print(f"Creating environment: {env_id}")
env = make_vec_env(env_id, n_envs=1)

Creating environment: PandaPickAndPlace-v3


In [5]:
#=========================================================================
# The observation space for this environment is a dictionary,
# so we must use 'MultiInputPolicy'.
# TD3 (Twin Delayed DDPG) is an improvement over DDPG that
# mitigates overestimation bias in the Q-function.
# We'll also use a TensorBoard log to visualize the training process.
model = TD3(
    policy="MultiInputPolicy",
    env=env,
    verbose=0, # Set verbose to 1 to see a progress bar and training logs
    device="cuda",  # ✅ Use GPU
    # tensorboard_log="./td3_panda_tensorboard/"
    # ==============================================================================
# 2. Model Initialization
# =====tensorboard_log="./td3_panda_tensorboard/"
)


In [6]:
# ==============================================================================
# 3. Training the Model
# ==============================================================================
# The original training time (100,000) is too short for this complex task.
# A more realistic starting point is 1,000,000 timesteps.
# The 'progress_bar' is a helpful feature to track the training progress.
total_timesteps = 1_000_000
print(f"Starting to train the model for {total_timesteps} timesteps...")
model.learn(total_timesteps=total_timesteps, progress_bar=True)
print("Training finished.")


Output()

Starting to train the model for 1000000 timesteps...


Training finished.


In [7]:
# ==============================================================================
# 4. Saving the Trained Model
# ==============================================================================
# You will need to mount your Google Drive in Colab to save the model.
# The user's original path is used here.
model_save_path = "./td3_panda_pick_and_place"
print(f"Saving the model to {model_save_path}")
model.save(model_save_path)

Saving the model to ./td3_panda_pick_and_place


In [8]:
# ==============================================================================
# 5. Evaluation
# ==============================================================================
# Create a new environment for evaluation with 'human' render mode.
# This allows you to visualize the agent's performance.
# We use DummyVecEnv to wrap the single evaluation environment for compatibility.
eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="human")])

print("Starting evaluation...")
# Evaluate the trained agent for a number of episodes.
# `render=True` will show the environment window.
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, render=True)

print("Evaluation finished.")
print(f"Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")
 
# Close the evaluation environment
eval_env.close()

Starting evaluation...


c:\Users\user\anaconda3\envs\sb3_env\lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Evaluation finished.
Mean reward: -50.00 ± 0.00


In [1]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.get_device_name(0))  # Show GPU name


True
NVIDIA GeForce GTX 1050 Ti


In [2]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
